<a href="https://www.kaggle.com/code/mrafraim/dl-day-42-cnn-early-stopping-checkpointing?scriptVersionId=305133593" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Day 42: CNN Early Stopping & Checkpointing
*Validation-Driven Stopping • Best-Model Saving*

Welcome to Day 42!

Today You’ll Learn

1. Why training too long hurts generalization  
2. What early stopping is and how it works  
3. The patience strategy for stable stopping  
4. Model checkpointing during training  
5. Saving the best validation model  
6. A practical CNN training loop implementation

If you found this notebook helpful, your **<b style="color:skyblue;">UPVOTE</b>** would be greatly appreciated! It helps others discover the work and supports continuous improvement.

---

# The Overfitting Problem

## Training Behavior in Deep Learning

During neural network training we often observe:

• Training loss steadily decreases  
• Validation loss decreases initially  
• Eventually validation loss begins increasing  

This phenomenon indicates overfitting.

## Why Overfitting Happens

Deep neural networks have very high capacity.

After learning meaningful patterns, the model may start:

• memorizing noise  
• memorizing training samples  
• over-specializing to the training distribution

## Typical Training Curve

Example progression:

Epoch 1–10  
Model learns general features.

Epoch 10–20  
Model reaches optimal generalization.

Epoch 20+  
Model begins memorizing training data.

## Key Insight

The best model is rarely the final epoch.

Instead, it occurs at the point where validation performance peaks.

Therefore training must detect that moment automatically.

# Early Stopping

Early stopping is a regularization technique that stops training when
validation performance stops improving.

This prevents the model from continuing into the overfitting region.

## Basic Rule

Training stops when:

> Validation metric does not improve for several consecutive epochs.

## Example

Validation loss across epochs:

Epoch 10 → 0.42  
Epoch 11 → 0.39  
Epoch 12 → 0.37  ← best model  
Epoch 13 → 0.38  
Epoch 14 → 0.40  
Epoch 15 → 0.41  

The optimal model occurs at **epoch 12**.

Training should stop shortly afterward.

# Patience Strategy

## Why Patience Is Needed

Validation metrics fluctuate due to:

• random minibatches  
• stochastic optimization  
• data variability  

Stopping immediately after one worse epoch would be unstable.

## Patience Parameter

Patience specifies how many epochs we wait
before stopping after the last improvement.

Example:

patience = 5

## Example Behavior

Epoch 12 → best validation score  

Epoch 13 → worse  
Epoch 14 → worse  
Epoch 15 → worse  
Epoch 16 → worse  
Epoch 17 → worse  

Since 5 epochs passed without improvement → stop training.

# Model Checkpointing

Model checkpointing is a standard practice in deep learning training pipelines.  
It ensures that the best-performing model and training state are preserved, even if training continues or crashes.

Without checkpointing, you risk losing the best model due to overfitting in later epochs or unexpected interruptions.

## Why Checkpoints Are Critical

During training, the model may reach its best validation performance before the final epoch.

Example:

| Epoch | Validation Loss |
|------|------|
| 8 | 0.45 |
| 9 | 0.40 |
| 10 | 0.36 ← best |
| 11 | 0.38 |
| 12 | 0.41 |

If we only keep the final model, performance would be worse.

Therefore:

> Save the model whenever validation performance improves.

This guarantees that the best model is preserved.

## Basic Checkpoint Example

Saving only the model weights:

```python
torch.save(model.state_dict(), "best_model.pth")
```

This stores the **parameters of the network**.

## Loading the Model Later

To restore the model:

```python
model.load_state_dict(torch.load("best_model.pth"))
```

This loads the saved weights into the architecture.

Important rule:

>The model architecture must match exactly when loading weights.

## Professional Checkpointing

In real ML training pipelines, saving only weights is not sufficient.

A proper checkpoint usually includes:

• model weights<br>
• optimizer state<br>
• learning rate scheduler state<br>
• epoch number

This allows training to resume exactly where it stopped.

Example checkpoint structure:

```python
checkpoint = {
    "epoch": epoch,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "scheduler_state": scheduler.state_dict()
}

torch.save(checkpoint, "checkpoint.pth")
```

## Reloading a Full Checkpoint

To resume training from a saved checkpoint:

```python
checkpoint = torch.load("checkpoint.pth")

model.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optimizer_state"])
scheduler.load_state_dict(checkpoint["scheduler_state"])

start_epoch = checkpoint["epoch"]
```

Now training can continue from the saved state.

## Best Practices in Real Training Pipelines

Typical deep learning training systems maintain multiple checkpoints:

| Checkpoint Type       | Purpose                         |
| --------------------- | ------------------------------- |
| best_model.pth        | Best validation performance     |
| latest_checkpoint.pth | Most recent training state      |
| periodic checkpoints  | Recovery for long training jobs |

Example strategy:

```
Save best model when validation improves
Save checkpoint every N epochs
Keep latest checkpoint for crash recovery
```


# Practical Early Stopping Logic

This is a production-style implementation of early stopping - simple, effective, and widely used.

It combines:

- **validation monitoring**
- **best model saving (checkpointing)**
- **training termination logic**


> Track the **best validation loss** and stop training when the model **fails to improve for a fixed number of epochs (patience)**.


### Key Variables

```python
best_val_loss = float("inf")   # best observed validation loss
patience = 5                   # allowed epochs without improvement
counter = 0                    # tracks how many epochs since last improvement
```

## Training Loop with Early Stopping

```python

best_val_loss = float("inf")
patience = 5
counter = 0

for epoch in range(num_epochs):

    train_loss = train_one_epoch(model, train_loader, optimizer)
    val_loss = validate(model, val_loader)

    print(f"Epoch {epoch} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # Check improvement
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        counter = 0

        # Save best model
        torch.save(model.state_dict(), "best_model.pth")

    else:
        counter += 1

        if counter >= patience:
            print("Early stopping triggered")
            break

```

## How It Works (Step-by-Step)

1. Model improves → reset counter
2. Model does not improve → increment counter
3. If no improvement for `patience` epochs → stop training

Example:

| Epoch | Val Loss | Counter  |
| ----- | -------- | -------- |
| 10    | 0.37     | 0 (best) |
| 11    | 0.38     | 1        |
| 12    | 0.39     | 2        |
| 13    | 0.40     | 3        |
| 14    | 0.41     | 4        |
| 15    | 0.42     | 5 → STOP |


# Choosing the Monitoring Metric

Early stopping and checkpointing are only as good as the metric you monitor.  
Choosing the wrong metric leads to optimizing the wrong behavior, a very common real-world mistake.

> You are not optimizing the model, you are optimizing the metric.


## What This Decision Controls

The chosen validation metric determines:

- **when training stops**
- **which model gets saved**
- **what “best model” actually means**

If the metric is misaligned with the task, your model will be technically optimal but practically useless.

## Common Monitoring Metrics

### Validation Loss

- Most commonly used
- Smooth and stable signal
- Works well during early training

**Best for:**

- general training stability
- when no specific metric requirement exists

**Limitation:**

- Lower loss does not always mean better real-world performance

### Validation Accuracy

- Simple and intuitive
- Easy to interpret

**Best for:**

- balanced datasets
- basic classification tasks

**Limitation:**

- Misleading for imbalanced datasets
- Ignores class-wise performance

### F1 Score

Balances:

- Precision
- Recall

$$
F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}
$$

**Best for:**

- imbalanced datasets
- multi-label classification
- real-world decision systems

Why practitioners prefer it:

> It reflects both **false positives and false negatives**


### mAP (Mean Average Precision)

Used in:

- object detection
- multi-label ranking problems

Measures:

- how well predictions are ranked and localized

**Best for:**

- detection models (YOLO, Faster R-CNN, etc.)

### Dice Score / IoU

Used in:

- image segmentation

Measures:

- overlap between predicted mask and ground truth

$$
Dice = \frac{2TP}{2TP + FP + FN}
$$

**Best for:**

- medical imaging
- pixel-wise prediction tasks


## Typical Industry Practice

| Task | Metric to Monitor |
|------|------------------|
| Image Classification | Validation Loss / F1 |
| Multi-label Classification | F1 (micro/macro) |
| Object Detection | mAP |
| Segmentation | Dice / IoU |

# Common Mistakes

• Saving only the final model  
• Monitoring training loss instead of validation loss  
• Using very small patience values  
• Forgetting to reload the best checkpoint  
• Not saving optimizer state when resuming training

# Real Training Pipeline

Step 1  
Train model for one epoch.

Step 2  
Compute validation metric.

Step 3  
If metric improves → save checkpoint.

Step 4  
If no improvement for N epochs → stop training.

Step 5  
Reload best checkpoint for final evaluation.

# Key Takeaways from Day 42

• Best model usually appears before final epoch  
• Early stopping prevents overfitting  
• Patience avoids unstable stopping decisions  
• Checkpointing preserves the best model  
• Always evaluate using the saved best checkpoint

---

<p style="text-align:center; color:skyblue; font-size:18px;">
© 2026 Mostafizur Rahman
</p>
